In [ ]:
#CELULA 1: Imports, Variaveis de Ambiente, GCP Client e Configuracoes Globais

import os
import requests
import pandas as pd
from datetime import datetime, timedelta
from google.cloud import storage

# 1. Autenticação Local GCP
BASE_DIR = os.path.dirname(os.path.abspath(""))
KEY_PATH = os.path.join(BASE_DIR, "config", "gcp_key.json")
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = KEY_PATH

# 2. Instanciação do Client do Google Cloud Storage
client = storage.Client.from_service_account_json(KEY_PATH)

# 3. Parâmetros Globais (Regra D-1)
ANO_ALVO = "2026"
BUCKET_BRONZE = "cvm-landing-zone-bronze-dev"

HOJE = datetime.now().date()
D1_LIMITE = HOJE - timedelta(days=1)
MES_LIMITE_D1 = D1_LIMITE.month

bucket = client.bucket(BUCKET_BRONZE)

print(f"✔ Client GCP inicializado para CDA_FI!")
print(f"✔ Target Bronze Bucket: gs://{BUCKET_BRONZE}")
print(f"✔ Data Limite (D-1): {D1_LIMITE.strftime('%Y-%m-%d')}")
print(f"✔ Mês limite a ser verificado: {MES_LIMITE_D1:02d}/{ANO_ALVO}")

In [ ]:
#CELULA 2: Ingestao Incremental com Janela Movel de 90 Dias (Sigilo CVM) para CDA_FI

def upload_in_memory(url: str, gcs_path: str, content_type: str) -> bool:
    """Faz o streaming do download direto para o GCS em memória."""
    try:
        response = requests.get(url, timeout=90)
        if response.status_code == 200:
            blob = bucket.blob(gcs_path)
            blob.upload_from_string(response.content, content_type=content_type)
            print(f" Success: gs://{BUCKET_BRONZE}/{gcs_path}")
            return True
        elif response.status_code == 404:
            print(f" Arquivo ainda não publicado pela CVM (HTTP 404): {url}")
            return False
        else:
            print(f" Erro HTTP {response.status_code} para {url}")
            return False
    except Exception as e:
        print(f" Exceção ao processar {url}: {str(e)}")
        return False

def executar_ingestao_cda_fi_janela_movel():
    print(f"\n=== INICIANDO INGESTÃO DE CARTEIRAS (CDA_FI) COM JANELA MÓVEL DE 90 DIAS ===")
    hoje_date = datetime.now().date()
    
    # Define a janela móvel dos últimos 3 meses (M, M-1, M-2) a partir de D-1
    # Exemplo: Se D-1 for no Mês 10, a janela móvel engloba os meses 08, 09 e 10
    mes_limite_d1 = MES_LIMITE_D1
    mes_inicio_janela = max(1, mes_limite_d1 - 2)
    
    print(f"✔ Mês Limite D-1: {mes_limite_d1:02d}/{ANO_ALVO}")
    print(f"✔ Janela Móvel de Reavaliação (Sigilo 90 dias): Meses {mes_inicio_janela:02d} até {mes_limite_d1:02d} de {ANO_ALVO}\n")

    for mes in range(1, mes_limite_d1 + 1):
        mes_str = f"{mes:02d}"
        ano_mes = f"{ANO_ALVO}{mes_str}"
        
        path_cda_gcs = f"cvm/cda_fi/ano={ANO_ALVO}/mes={mes_str}/cda_fi_{ano_mes}.zip"
        blob = bucket.blob(path_cda_gcs)
        
        existe_no_gcs = blob.exists()
        esta_na_janela_movel = (mes >= mes_inicio_janela)
        
        # 1. Mês histórico fora da janela de 90 dias (Mês < M-2) e já existente -> Imutável
        if existe_no_gcs and not esta_na_janela_movel:
            print(f"✔ [CDA_FI] Mês {mes_str}/{ANO_ALVO} consolidado (fora da janela de sigilo). Pulando...")
            continue

        # 2. Mês dentro da janela dos 90 dias e já verificado HOJE -> Evita re-downloads no mesmo dia
        if existe_no_gcs and esta_na_janela_movel:
            blob.reload()
            if blob.updated.date() == hoje_date:
                print(f"✔ [CDA_FI] Mês {mes_str}/{ANO_ALVO} (Janela 90d) já foi atualizado HOJE ({hoje_date}). Pulando...")
                continue
            else:
                print(f"🔄 [CDA_FI] Mês {mes_str}/{ANO_ALVO} dentro da janela de 90d. Atualizando novos sigilos expirados...")

        # 3. Executa o download/atualização
        if not existe_no_gcs:
            print(f"🚀 [CDA_FI] Baixando novo arquivo de carteira: {mes_str}/{ANO_ALVO}...")

        url_cda = f"http://dados.cvm.gov.br/dados/FI/DOC/CDA/DADOS/cda_fi_{ano_mes}.zip"
        upload_in_memory(url_cda, path_cda_gcs, "application/zip")

# Executa a ingestão com a regra de 90 dias
executar_ingestao_cda_fi_janela_movel()